In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch, Dataset
from tqdm import tqdm
import sys

sys.path.append("../")

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_with_layer_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_with_layer_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"

sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)

X_pixel = np.concatenate([sig_pixel_spacetime, bg_pixel_spacetime], axis=0)
X_mppc = np.concatenate([sig_mppc_spacetime, bg_mppc_spacetime], axis=0)
y = np.concatenate(
    [np.ones(sig_pixel_spacetime.shape[0]), np.zeros(bg_pixel_spacetime.shape[0])], axis=0
)

from sklearn.model_selection import train_test_split
X_pixel_train, X_pixel_val, X_mppc_train, X_mppc_val, y_train, y_val = train_test_split(
    X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
import src.torch.pre_processing.graph_batching as graph_batching
from importlib import reload
reload(graph_batching)

graph_builder = graph_batching.CombinedGraphBuilder(connect_layers=True, mppc_timing_cutoff=0.1)
event_processor = graph_batching.EventProcessor(graph_builder=graph_builder)

In [ ]:
event_index = 4
test_graph = event_processor.process_single_event(X_pixel=X_pixel_train[event_index], X_mppc=X_mppc_train[event_index], labels=y_train[event_index])
test_graph